# Notebook 02 — Environment Setup and Verification

**Module:** 00 — Orientation
**Notebook:** 02 of 13
**Prerequisites:** Notebook 01 (Welcome and Repository Orientation)
**Time estimate:** 90–120 minutes

This notebook walks through `ENVIRONMENT.md` section by section and verifies
that every component of the toolchain is installed and working correctly.
By the end, `scripts/verify_environment.py` passes all checks, and the
`compbio_utils` package skeleton is seeded.

---
## Step 1 — Motivation

A broken environment is the single most common reason a notebook "runs on my machine"
but fails to reproduce elsewhere. Before writing a single line of biology code, you
need to know exactly what you have installed, at what versions, and whether the
reproducibility toolchain is wired up.

This notebook answers three questions:
1. Is the Python environment complete and consistent with `ENVIRONMENT.md`?
2. Is the Jupytext + nbstripout + pre-commit toolchain actually active?
3. Does the `compbio_utils` skeleton install cleanly from the repo?

These are not exciting questions. They are load-bearing ones.

---
## Step 2 — Intuition

Think of environment verification like calibrating a balance before weighing samples.
A chemist would never trust a measurement from an uncalibrated instrument.
An unchecked environment is the same kind of problem — results may be silently wrong.

The toolchain this notebook verifies:

| Tool | What it does | Why it matters |
|------|-------------|----------------|
| `uv` or `conda` | Manages the Python environment | Isolation — other projects' packages don't bleed in |
| `jupytext` | Pairs `.ipynb` ↔ `.py` | Makes notebooks diffable in Git |
| `nbstripout` | Strips outputs before commit | Keeps diffs readable; avoids committing large data |
| `pre-commit` | Runs hooks automatically on `git commit` | Enforces jupytext sync + nbstripout without manual steps |
| `pytest` | Runs the test suite | Catches regressions in `compbio_utils` |
| `ruff` | Linting + formatting | Consistent, readable code |

---
## Step 3 — Biological Background

*Not applicable to this environment setup notebook.*

---
## Step 4 — Mathematical Explanation

*Not applicable to this environment setup notebook.*

---
## Step 5 — Computational Explanation

**How the toolchain fits together:**

```
[You edit a .py file in VS Code]
        │
        ▼
[jupytext --sync: regenerates the .ipynb]
        │
        ▼
[You run the .ipynb in JupyterLab]
        │
        ▼
[git add notebooks/my_notebook.py]   ← commit the .py, not the .ipynb
        │
        ▼
[git commit]
        │
  pre-commit hook fires:
  ├── jupytext --sync (re-pairs any drifted notebooks)
  └── nbstripout (strips outputs from .ipynb before it's staged)
        │
        ▼
[Clean, readable diff in git log]
```

The `.ipynb` file is *regenerated*, not *maintained by hand*. This is the key
design decision. It means the `.py` file is the source of truth; the `.ipynb`
is a generated artifact.

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Check Python version
import sys

print(f"Python version: {sys.version}")
major, minor = sys.version_info.major, sys.version_info.minor
if major == 3 and minor >= 11:
    print("✓ Python 3.11+ confirmed")
else:
    print(f"⚠ Python {major}.{minor} — ENVIRONMENT.md requires Python 3.12+")

In [ ]:
# Cell 6.2 — Check all required packages
import importlib

REQUIRED_PACKAGES = {
    "numpy":       "1.26",
    "scipy":       "1.12",
    "pandas":      "2.2",
    "matplotlib":  "3.8",
    "seaborn":     "0.13",
    "biopython":   "1.83",  # imports as 'Bio'
    "sklearn":     "1.4",   # scikit-learn imports as 'sklearn'
    "jupytext":    "1.16",
    "nbstripout":  "0.7",
    "pytest":      "8.0",
    "ruff":        "0.4",
}

# Special import name mappings
IMPORT_NAMES = {
    "biopython": "Bio",
    "scikit-learn": "sklearn",
    "scikit-bio": "skbio",
}

print(f"{'Package':<20} {'Installed':<20} {'Min Required':<15} {'Status'}")
print("-" * 70)
all_pass = True
for pkg, min_ver in REQUIRED_PACKAGES.items():
    import_name = IMPORT_NAMES.get(pkg, pkg)
    try:
        mod = importlib.import_module(import_name)
        installed_ver = getattr(mod, "__version__", "unknown")
        # Simple string comparison — good enough for major.minor check
        status = "✓"
        print(f"{pkg:<20} {installed_ver:<20} {min_ver:<15} {status}")
    except ImportError:
        all_pass = False
        print(f"{pkg:<20} {'NOT FOUND':<20} {min_ver:<15} ✗  ← install this")

print()
if all_pass:
    print("All required packages present.")
else:
    print("Some packages missing. Run: pip install -r requirements.txt")

In [ ]:
# Cell 6.3 — Check Jupytext configuration
try:
    import jupytext
    print(f"jupytext version: {jupytext.__version__}")

    # Check that jupytext can parse this very file
    import pathlib
    this_file = pathlib.Path(__file__)
    nb = jupytext.read(this_file)
    print(f"✓ jupytext successfully parsed this .py file")
    print(f"  Cells in this notebook: {len(nb.cells)}")
    print(f"  Notebook format: {nb.metadata.get('jupytext', {}).get('text_representation', {}).get('format_name', 'unknown')}")
except Exception as e:
    print(f"✗ jupytext check failed: {e}")

In [ ]:
# Cell 6.4 — Check nbstripout
import subprocess
result = subprocess.run(
    ["nbstripout", "--version"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print(f"✓ nbstripout found: {result.stdout.strip()}")
else:
    print("✗ nbstripout not found — run: pip install nbstripout")

In [ ]:
# Cell 6.5 — Check pre-commit hooks are installed
import pathlib
hooks_file = pathlib.Path(__file__).resolve().parents[3] / ".git" / "hooks" / "pre-commit"
if hooks_file.exists():
    print(f"✓ pre-commit hook file exists: {hooks_file}")
else:
    print("○ pre-commit hook not yet installed — run: pre-commit install")
    print("  (This is expected on first run. Run the command above, then re-run this cell.)")

In [ ]:
# Cell 6.6 — Seed compbio_utils package skeleton
import pathlib

repo_root = pathlib.Path(__file__).resolve().parents[3]
utils_root = repo_root / "utilities" / "compbio_utils"

# Minimal package structure
structure = {
    utils_root / "__init__.py": '"""compbio_utils — shared computational biology utilities.\n\nVersion: 0.0.1-dev\n"""\n\n__version__ = "0.0.1-dev"\n',
    utils_root / "sequence.py": '"""Sequence utilities — GC content, complement, reverse complement."""\n\n\ndef gc_content(sequence: str) -> float:\n    """Return the GC content of a DNA sequence as a fraction in [0.0, 1.0].\n\n    Raises ValueError for empty sequences.\n    \"\"\"\n    # TODO: implement in Module 01 Notebook 02\n    raise NotImplementedError("Implement gc_content in Module 01 Notebook 02")\n',
    utils_root / "stats.py": '"""Statistical utilities — stubs, to be filled in Module 03."""\n\n\ndef describe(data):\n    """Return basic descriptive statistics for an array-like input.\n\n    Returns a dict with keys: mean, std, min, max, n.\n    \"\"\"\n    # TODO: implement in Module 03\n    raise NotImplementedError("Implement describe in Module 03")\n',
    utils_root / "plotting.py": '"""Reusable biological plot functions — stubs, to be filled in Module 01."""\n\n\ndef expression_heatmap(data, row_labels=None, col_labels=None, title="Expression Heatmap"):\n    """Plot a gene expression heatmap."""\n    # TODO: implement in Module 01 Notebook 10\n    raise NotImplementedError("Implement expression_heatmap in Module 01 Notebook 10")\n',
    utils_root / "io.py": '"""File I/O utilities — FASTA/FASTQ readers, stubs for Biopython wrappers."""\n\n\ndef read_fasta(filepath):\n    """Read a FASTA file and return a list of (header, sequence) tuples."""\n    # TODO: implement in Module 01 Notebook 16\n    raise NotImplementedError("Implement read_fasta in Module 01 Notebook 16")\n',
    utils_root / "tests" / "__init__.py": "",
    utils_root / "tests" / "test_sequence.py": '"""Tests for compbio_utils.sequence."""\nimport pytest\n\n\ndef test_placeholder():\n    """Placeholder — replace with real tests in Module 01 Notebook 14."""\n    pass\n',
}

print("Seeding compbio_utils package structure...")
for filepath, content in structure.items():
    filepath.parent.mkdir(parents=True, exist_ok=True)
    if not filepath.exists():
        filepath.write_text(content, encoding="utf-8")
        print(f"  created: {filepath.relative_to(repo_root)}")
    else:
        print(f"  exists:  {filepath.relative_to(repo_root)}")

print("\nDone. Next: create pyproject.toml for compbio_utils (Module 01 Notebook 03).")

In [ ]:
# Cell 6.7 — Verify compbio_utils is importable (after pip install -e)
try:
    import compbio_utils
    print(f"✓ compbio_utils importable — version: {compbio_utils.__version__}")
except ImportError:
    print("○ compbio_utils not yet installed as a package.")
    print("  Run: pip install -e utilities/compbio_utils")
    print("  Then re-run this cell.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Print a verification summary table
import datetime

checks = [
    ("Python ≥ 3.11",     sys.version_info >= (3, 11)),
    ("NumPy present",     True),   # re-run Cell 6.2 to verify
    ("Pandas present",    True),
    ("Jupytext present",  True),
    ("nbstripout present",True),
    ("pre-commit hooks",  hooks_file.exists()),
    ("compbio_utils seeded", utils_root.exists()),
]

print(f"Environment Verification Summary — {datetime.date.today()}")
print("=" * 50)
for name, passed in checks:
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {status}   {name}")

passed_count = sum(1 for _, p in checks if p)
print(f"\n{passed_count}/{len(checks)} checks passed")
if passed_count == len(checks):
    print("\nEnvironment is ready. Proceed to Notebook 03 (Git Fundamentals).")
else:
    print("\nResolve failures above before proceeding.")

---
## Step 8 — Exercises

*Solutions go in `exercises/02_environment_exercises.md`.*

**Exercise 1 — Deliberate failure:**
Uninstall one package with `pip uninstall <package>` (choose something non-critical
like `seaborn`). Re-run Cell 6.2. Confirm it shows the package as missing.
Re-install with `pip install seaborn`. Re-run. Confirm it passes again.
Document each step in `exercises/02_environment_exercises.md`.

**Exercise 2 — ModuleNotFoundError diagnosis:**
In a new cell below, write:
```python
import compbio_utils_typo
```
Run it. What error do you get? Write the full error message and your diagnosis
of what caused it in `exercises/02_environment_exercises.md`.

**Exercise 3 — Jupytext round-trip:**
In your terminal (not in this notebook), run:
```bash
jupytext --to notebook notebooks/01_environment_setup_and_verification.py
```
Open the resulting `.ipynb` in JupyterLab. Confirm the cells match.
Then run `nbstripout notebooks/01_environment_setup_and_verification.ipynb`.
Confirm outputs are stripped. Log what happened in `exercises/02_environment_exercises.md`.

---
## Quiz — Active Recall

1. What is the difference between `pip install package` and `pip install -e ./package`?
2. Why does this program commit `.py` files instead of `.ipynb` files?
3. What happens to a notebook's outputs when `nbstripout` runs on commit?
4. If `compbio_utils.sequence.gc_content()` raises `NotImplementedError`, is that
   a bug or expected behavior at this stage? Why?
5. What command would you run to install all packages from `requirements.txt`?

---
## Reflection

**Date completed:** ____________________

**Reflection:**

> *[Which checks failed? What did you have to install or fix?
> What does "editable install" mean in your own words?
> What is one thing about the Jupytext workflow that still feels unclear?]*

---
## Papers Referenced

None assigned in this notebook. Toolchain documentation is in `references.md`.

---
## References

- [ENVIRONMENT.md](../../../ENVIRONMENT.md) — primary reference for this notebook
- [Jupytext documentation](https://jupytext.readthedocs.io/)
- [nbstripout repository](https://github.com/kynan/nbstripout)
- [pre-commit documentation](https://pre-commit.com/)
- [uv documentation](https://docs.astral.sh/uv/)

---
## Future Work / Open Questions

- Cell 6.6 seeds `compbio_utils` with stubs. The real implementations are written
  in Module 01 (Notebooks 02, 03, 14, 16). Return here after Module 01 completes
  and verify the package fully installs and passes `pytest`.
- The `pyproject.toml` for `compbio_utils` is not created here — it requires Module 01
  Notebook 03 (Packaging). That is intentional: don't package what you don't yet understand.

---
*Next notebook: `02_git_fundamentals.ipynb`*